<a href="https://colab.research.google.com/github/agtorres1/DDS_tp_anual/blob/main/notebooks/02_autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!pip install kaggle
!kaggle datasets download -d mlg-ulb/creditcardfraud
!unzip -o creditcardfraud.zip

Dataset URL: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
License(s): DbCL-1.0
100% 66.0M/66.0M [00:00<00:00, 158MB/s]

Archive:  creditcardfraud.zip
  inflating: creditcard.csv          


## 1. Carga y preprocesamiento de datos

Mismo proceso que en el MLP: escalado de Time y Amount, separación de
X e y, y split train/test con stratify.

In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('creditcard.csv')

scaler = StandardScaler()
df['Time'] = scaler.fit_transform(df[['Time']])
df['Amount'] = scaler.fit_transform(df[['Amount']])

X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (227845, 30) Test: (56962, 30)


## 3. Filtrar solo transacciones legítimas para entrenar

A diferencia del MLP, el Autoencoder se entrena SOLO con transacciones
legítimas, para que aprenda el patrón de "lo normal".

In [5]:
X_train_legitimas = X_train[y_train == 0]

print("Transacciones legítimas para entrenar:", X_train_legitimas.shape)

Transacciones legítimas para entrenar: (227451, 30)


## 4. Arquitectura del Autoencoder

Un Autoencoder tiene dos partes:
- Encoder: comprime la entrada a una representación más chica
- Decoder: intenta reconstruir la entrada original a partir de esa versión comprimida

La idea es que, al entrenarlo solo con transacciones normales, va a
volverse muy bueno reconstruyendo transacciones normales, pero malo
reconstruyendo fraudes (porque nunca los vio).

In [6]:
from tensorflow import keras
from tensorflow.keras import layers

n_features = X_train.shape[1]

autoencoder = keras.Sequential([
    layers.Input(shape=(n_features,)),
    # Encoder: va comprimiendo
    layers.Dense(20, activation='relu'),
    layers.Dense(14, activation='relu'),
    layers.Dense(7, activation='relu'),   # cuello de botella
    # Decoder: va reconstruyendo
    layers.Dense(14, activation='relu'),
    layers.Dense(20, activation='relu'),
    layers.Dense(n_features, activation='linear')
])

autoencoder.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 20)             │           620 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 14)             │           294 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           105 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 14)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 30)             │           630 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,061 (8.05 KB)

 Trainable params: 2,061 (8.05 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Compilación y entrenamiento

El objetivo del Autoencoder es que la salida sea lo más parecida
posible a la entrada, por eso usamos MSE como función de pérdida:
mide qué tan lejos está la reconstrucción del original.

In [ ]:
autoencoder.compile(optimizer='adam', loss='mse')

historia = autoencoder.fit(
    X_train_legitimas, X_train_legitimas,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    shuffle=True
)

Epoch 1/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.6769 - val_loss: 0.4584
Epoch 2/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3953 - val_loss: 0.3483
Epoch 3/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3333 - val_loss: 0.3167
Epoch 4/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.3113 - val_loss: 0.3027
Epoch 5/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.3000 - val_loss: 0.2932
Epoch 6/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.2907 - val_loss: 0.2848
Epoch 7/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2839 - val_loss: 0.2803
Epoch 8/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2767 - val_loss: 0.2832
Epoch 9/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2732 - val_loss: 0.2685
Epoch 10/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.2672 - val_loss: 0.2624
Epoch 11/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 0.2616 - val_loss: 0.2580
Epoch 12/20
711/711 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

## 6. Calcular el error de reconstrucción sobre el conjunto de test

Le pasamos TODO el conjunto de test (con fraude y sin fraude) y medimos
qué tan bien reconstruye cada transacción. Esperamos que las fraudulentas
tengan un error mucho más alto.

In [ ]:
import numpy as np

X_test_reconstruido = autoencoder.predict(X_test)

error_reconstruccion = np.mean(np.square(X_test.values - X_test_reconstruido), axis=1)

errores_df = pd.DataFrame({
    'error': error_reconstruccion,
    'clase_real': y_test.values
})

errores_df.groupby('clase_real')['error'].describe()